In [ ]:
# 1. INSTALACIÓN 
!pip install -q -U bitsandbytes
!pip install -q transformers==4.44.0 datasets peft==0.12.0 trl==0.9.6 accelerate==0.34.0


In [ ]:
import bitsandbytes as bnb
import torch
print(torch.cuda.is_available())        # debe ser True
print(torch.cuda.get_device_name(0))    # debe decir Tesla T4
print(bnb.__version__)

True
Tesla T4
0.49.2


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 2. CONFIGURACIÓN
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
dataset_id = "Belenmrqz/genz-marketing-prompts"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# 3. LORA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 4. TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = 512

# 5. DATASET
dataset = load_dataset(dataset_id, split="train")

def format_chat(example):
    example['text'] = tokenizer.apply_chat_template(
        example['messages'], tokenize=False
    )
    return example

dataset = dataset.map(format_chat)

# 6. TRAINING ARGS 
# sin fp16 ni bf16, bitsandbytes gestiona la precisión
training_arguments = SFTConfig(
    output_dir="./genz-model-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=5,
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=False,
    dataset_text_field="text",
    packing=False,
)

# 7. TRAINER
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_arguments,
    tokenizer=tokenizer,  # ✅ en trl 0.9.6 se llama tokenizer, no processing_class
)

print("🚀 ¡Empezando el entrenamiento! Vibes are immaculate...")
trainer.train()

trainer.model.save_pretrained("./genz-adapter")
print("✅ ¡Modelo ajustado y guardado! No cap.")

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:289: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 512
  warnings.warn(


Map:   0%|          | 0/12 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


🚀 ¡Empezando el entrenamiento! Vibes are immaculate...


...

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


✅ ¡Modelo ajustado y guardado! No cap.
